# AI Career Assistant Team

| Agent          | Job                      |
| -------------- | ------------------------ |
| Research Agent | Finds skills required    |
| Roadmap Agent  | Creates learning roadmap |
| Critic Agent   | Reviews roadmap          |
| Supervisor     | Coordinates everything   |


User Query
    |
    v
Supervisor
    |
    +--> Research Agent
    |        |
    |        v
    |   Required Skills
    |
    +--> Roadmap Agent
    |        |
    |        v
    |   Learning Plan
    |
    +--> Critic Agent
             |
             v
      Improvements
             |
             v
       Final Response

```
User Query
    |
    v
Supervisor
    |
    +--> Research Agent
    |        |
    |        v
    |   Required Skills
    |
    +--> Roadmap Agent
    |        |
    |        v
    |   Learning Plan
    |
    +--> Critic Agent
             |
             v
      Improvements
             |
             v
       Final Response
```

In [ ]:
from rich import print
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

llm=init_chat_model(model="groq:meta-llama/llama-4-scout-17b-16e-instruct")

In [ ]:
from typing import TypedDict

class AgentState(TypedDict):
    user_query: str
    research: str
    roadmap: str
    critique: str
    final_answer: str

### Create Specialized Agents

#### Research Agent

In [ ]:
def research_agent(state: AgentState):
    query=state['user_query']

    prompt=f"""
        User goal: {query}
        List:
        - important skills
        - technologies
        - prerequisites

        keep concise.
    """

    response = llm.invoke(prompt)

    return {
        "research": response.content
    }

#### Roadmap Agent

In [ ]:
def roadmap_agent(state: AgentState):
    prompt=f"""
    Using this research:

    {state['research']}

    create a 6-month roadmap.
    """

    response = llm.invoke(prompt)

    return {
        "roadmap": response.content
    }

### Critic Agent

In [ ]:
def critic_agent(state: AgentState):
    prompt=f"""
    Review this roadmap:

    {state['roadmap']}

    Suggest improvements and missing topics.
    """

    response = llm.invoke(prompt)

    return {
        "critique": response.content
    }

#### Final Agent

In [ ]:
def final_agent(state: AgentState):

    final_response=f"""
    RESEARCH:
    {state['research']}

    ROADMAP:
    {state['roadmap']}

    CRITIQUE:
    {state['critique']}
    """

    return {
        "final_answer": final_response
    }

### Build LangGraph Workflow

In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

In [ ]:
graph.add_node("research",research_agent)
graph.add_node("roadmap",roadmap_agent)
graph.add_node("critic", critic_agent)
graph.add_node("final", final_agent)

In [ ]:
graph.set_entry_point("research")

graph.add_edge("research","roadmap")
graph.add_edge("roadmap", "critic")
graph.add_edge("critic", "final")

graph.add_edge("final", END)

In [ ]:
app = graph.compile()

In [ ]:
app

In [ ]:
result = app.invoke({"user_query":"I want to become an AI engineer"})

print(result)